# Visualize evaluation results

In [ ]:
from matplotlib import pyplot as plt
import ast
from pathlib import Path

import config

## Read results from files

In [ ]:
# Read results from files
infids = {}
times = {}
time_key = 'time'

for model_name in config.MODELS:
    basename = f'results/{model_name}'
    infids[model_name] = {pert: {} for pert in config.PERTS}
    times[model_name] = {}
    for pert in config.PERTS:
        infid_key = f'infid_{pert}'
        # Read ra results
        method_name = 'ra'
        filename = Path(f'{basename}_{method_name}.txt')
        if filename.is_file():
            with open(filename) as f:
                results = ast.literal_eval(f.read())
                infids[model_name][pert][method_name] = results[infid_key]
                times[model_name][method_name] = results[time_key]
        # Read ixg results
        method_name = 'ixg'
        filename = Path(f'{basename}_{method_name}.txt')
        if filename.is_file():
            with open(filename) as f:
                results = ast.literal_eval(f.read())
                infids[model_name][pert][method_name] = results[infid_key]
                times[model_name][method_name] = results[time_key]
        # Read ig results
        method_name = 'ig'
        for n_steps in config.N_STEPS:
            filename = Path(f'{basename}_{method_name}_nsteps{n_steps}.txt')
            if filename.is_file():
                with open(filename) as f:
                    results = ast.literal_eval(f.read())
                    method_name_full = method_name + str(n_steps)
                    infids[model_name][pert][method_name_full] = results[infid_key]
                    times[model_name][method_name_full] = results[time_key]
        # Read shap results
        method_name = 'shap'
        for n_samples in config.N_SAMPLES:
            filename = Path(f'{basename}_{method_name}_nsamples{n_samples}.txt')
            if filename.is_file():
                with open(filename) as f:
                    results = ast.literal_eval(f.read())
                    method_name_full = method_name + str(n_samples)
                    infids[model_name][pert][method_name_full] = results[infid_key]
                    times[model_name][method_name_full] = results[time_key]

## Plot results

In [ ]:
fig, axs = plt.subplots(4, 2, figsize=(10,16))
width = 0.75    # Bar width
alpha = 0.5     # Opacity
for i, model_name in enumerate(infids.keys()):
    # Plot infidelities
    # Plot infidelities for baseline perturbation
    pert = 'b'
    method_names = list(infids[model_name][pert].keys())
    infids_values = list(infids[model_name][pert].values())
    axs[0, i].barh(method_names, infids_values, width, facecolor='r', alpha=alpha)
    axs[0, i].set_xlim([0., 1.2 * max(infids_values)])
    axs[0, i].set_title(f"Infidelity of feature attributions on {model_name}")
    axs[0, i].set_xlabel(f"infid ({pert})")
    for idx, v in enumerate(infids_values):
        axs[0, i].text(v + 0.02 * v, idx, str(v), color='black')
    # Plot infidelities for noisy baseline perturbation
    pert = 'nb'
    method_names = list(infids[model_name][pert].keys())
    infids_values = list(infids[model_name][pert].values())
    axs[1, i].barh(method_names, infids_values, width, facecolor='r', alpha=alpha)
    axs[1, i].set_xlim([0., 1.2 * max(infids_values)])
    axs[1, i].set_xlabel(f"infid ({pert})")
    for idx, v in enumerate(infids_values):
        axs[1, i].text(v + 0.02 * v, idx, str(v), color='black')
    # Plot infidelities for noisy input perturbation
    pert = 'ni'
    method_names = list(infids[model_name][pert].keys())
    infids_values = list(infids[model_name][pert].values())
    axs[2, i].barh(method_names, infids_values, width, facecolor='r', alpha=alpha)
    axs[2, i].set_xlim([0., 1.2 * max(infids_values)])
    axs[2, i].set_xlabel(f"infid ({pert})")
    for idx, v in enumerate(infids_values):
        axs[2, i].text(v + 0.02 * v, idx, str(v), color='black')
    # Plot times
    method_names = list(times[model_name].keys())
    times_values = list(times[model_name].values())
    axs[3, i].barh(method_names, times_values, width, facecolor='b', alpha=alpha)
    axs[3, i].set_xlim([0., 1.2 * max(times_values)])
    axs[3, i].set_title(f"Runtime of feature attributions on {model_name}")
    axs[3, i].set_xlabel("runtime (in s)")
    for idx, v in enumerate(times_values):
        axs[3, i].text(v + 0.02 * v, idx, str(round(v, 2)), color='black')

fig.tight_layout()

In [ ]:
# Save results plot
fig.savefig(f"results/barplot.eps", format="eps")

In [ ]:
# Save results plot as png
fig.savefig(f"results/barplot.png", format="png")